In [0]:
import json
import re
import traceback
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, List, Optional
import pandas as pd

In [0]:

# 规则配置文件(修改到自己的目录)
CONFIG_PATH = Path("/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_UAT/FEIYI/json_compare_config.xlsx")
# 报告输出目录
REPORT_DIR = Path("/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_UAT/Report/")

In [0]:
# Kafka连接配置
KAFKA_BOOTSTRAP_SERVERS = "10.249.209.11:9092,10.249.209.13:9092,10.249.209.16:9092"

# topic映射定义：
# key   -> 标准topic（基准）
# value -> 新topic（被比较对象）
#
# 脚本会以key为主逐对处理，用同一条规则（input_cid1/input_uid1 查标准topic，
# input_cid2/input_uid2 查新topic）从两侧topic中检索消息。
topic_mapping = {
    # CBR topics
    'CBR_AU': 'CBR_AU_TS',
    'CBR_HK': 'CBR_HK_TS',
    'CBR_JP': 'CBR_JP_TS',
    'CBR_KR': 'CBR_KR_TS',
    'CBR_DrJart_KR': 'CBR_DrJart_KR_TS',
    'CBR_MY': 'CBR_MY_TS',
    'CBR_NZ': 'CBR_NZ_TS',
    'CBR_PH': 'CBR_PH_TS',
    'CBR_SG': 'CBR_SG_TS',
    'CBR_TH': 'CBR_TH_TS',
    'CBR_TW': 'CBR_TW_TS',
    'CBR_VN': 'CBR_VN_TS',
    'CBR_ID': 'CBR_ID_TS',
    # CBR Public topics
    'CBRPublic_AU': 'CBRPublic_AU_TS',
    'CBRPublic_HK': 'CBRPublic_HK_TS',
    'CBRPublic_JP': 'CBRPublic_JP_TS',
    'CBRPublic_KR': 'CBRPublic_KR_TS',
    'CBRPublic_MY': 'CBRPublic_MY_TS',
    'CBRPublic_NZ': 'CBRPublic_NZ_TS',
    'CBRPublic_PH': 'CBRPublic_PH_TS',
    'CBRPublic_SG': 'CBRPublic_SG_TS',
    'CBRPublic_TH': 'CBRPublic_TH_TS',
    'CBRPublic_TW': 'CBRPublic_TW_TS',
    'CBRPublic_VN': 'CBRPublic_VN_TS',
    'CBRPublic_ID': 'CBRPublic_ID_TS'
}

# consumer_timeout_ms：在for循环模式消费时无消息自动结束
KAFKA_CONSUMER_TIMEOUT_MS = 5000
# poll超时与空轮询上限：用于控制扫描topic时的等待时间
KAFKA_POLL_TIMEOUT_MS = 1000
KAFKA_MAX_EMPTY_POLLS = 3

# 规则字段映射路径
CID_JSON_PATH = "$.ConsumerBestRecord.BestRecord.SourceSystemList.SourceSystem[*].ConsumerId"
UID_JSON_PATH = "$.ConsumerBestRecord.BestRecord.Attributes.UniversalKey"

# 排除校验的字段
DEFAULT_EXCLUDED_PATHS = [
    # 随机生成, 不进行校验
    "$.Header.DocumentTimestamp",
    "$.Header.DocumentUUID",
    "$.ConsumerBestRecord.@RecordUUID",
    "$.ConsumerBestRecord.BestRecord.Attributes.RecordTimestamp",
    "$.ConsumerBestRecord.DerivedBestRecordList.DerivedBestRecord[*].Attributes.RecordTimestamp",

    "$.ConsumerBestRecord.BestRecord.Attributes.UniversalKey",
    #  Talend missed HobbyList
    "$.ConsumerBestRecord.DerivedBestRecordList.DerivedBestRecord[*].PersonalData.HobbyList",
    # Talend wrongly send as: "DoNotContact": "false", Databricks send to be same as other flag fields: "DoNotContact": false,
    "$.ConsumerBestRecord.DerivedBestRecordList.DerivedBestRecord[*].PersonalData.DoNotContact",
    # CustomerGroupList, talend only generated 1 value though there are 2 value in ConsumerList
    "$.ConsumerBestRecord.DerivedBestRecordList.DerivedBestRecord[*].CustomerGroupList.CustomerGroup",
]

# 无序数组按指定字段组合进行键匹配的配置
# key: 数组路径（使用 _normalize_path 后的形式，无 $ 前缀，数组索引统一为 [*]）
# value: dict
#   - fields: 用于组成匹配键的字段路径列表（相对于数组元素本身）
#   - preserve_null: 哪些字段的 None 应该保留为 None 参与匹配（null 只匹配 null）
UNORDERED_ARRAY_MATCH_KEYS = {
    "ConsumerBestRecord.DerivedBestRecordList.DerivedBestRecord": {
        "fields": [
            "@Level",
            "Attributes.BrandCode",
            "Attributes.DistributionChannelCode",
        ],
        "preserve_null": [
            "Attributes.DistributionChannelCode",
        ],
    }
}

# 是否将 JSON null 与空字符串 "" 视为等价（通过）
NULL_EQUIVALENT_TO_EMPTY_STRING = True

# 比对前需要对字符串值做 trim 的字段路径列表
TRIM_BEFORE_COMPARE_PATHS = [
    "$.ConsumerBestRecord.DerivedBestRecordList.DerivedBestRecord[*].ContactInformation.AddressList.Address.Address1",
    "$.ConsumerBestRecord.DerivedBestRecordList.DerivedBestRecord[*].ContactInformation.AddressList.Address[*].Address1",
]

# 差异报告列顺序（Excel/HTML 输出用）
DIFF_REPORT_COLUMNS = [
    'uid1', 'uid1-L1-RecordTimestamp', 'uid2', 'uid2-L1-RecordTimestamp',
    'Market', 'Brand', 'DistributionChannelCode', 'CID List 1', 'CID List 2',
    '字段路径', '差异类型',
    'JSON1值', 'JSON2值', 'JSON1类型', 'JSON2类型', '额外信息'
]

# 批量差异明细报告列顺序：在单条报告列前附加规则与 topic 信息
BATCH_DIFF_REPORT_COLUMNS = [
    'rule_id', 'input_cid1', 'input_cid2', 'input_uid1', 'input_uid2',
    'standard_topic', 'new_topic',
] + DIFF_REPORT_COLUMNS

In [0]:
"""
JSON比对脚本
按配置读取(input_cid1/input_uid1 与 input_cid2/input_uid2)规则，
从标准topic与新topic中提取最新匹配消息进行JSON差异比对。
"""
class SimplePrintLogger:
    """使用print输出日志，兼容logger.info/debug/warning/error/exception调用。"""

    @staticmethod
    def _format(msg: str, *args: Any) -> str:
        return msg % args if args else msg

    def info(self, msg: str, *args: Any) -> None:
        print("[Info] " + self._format(msg, *args))

    def debug(self, msg: str, *args: Any) -> None:
        print("[Debug] " + self._format(msg, *args))

    def warning(self, msg: str, *args: Any) -> None:
        print("[Warning] " + self._format(msg, *args))

    def error(self, msg: str, *args: Any) -> None:
        print("[Error] " + self._format(msg, *args))

    def exception(self, msg: str, *args: Any) -> None:
        print("[Error] " + self._format(msg, *args))
        print(traceback.format_exc().rstrip())

logger = SimplePrintLogger()

class JsonComparator:
    """JSON比对器"""
    
    def __init__(self, excluded_paths: List[str] = None, unordered_arrays: bool = False,
                 payload_std: Dict[str, Any] = None, payload_new: Dict[str, Any] = None,
                 unordered_array_match_keys: Optional[Dict[str, Any]] = None,
                 null_equivalent_to_empty: bool = True,
                 trim_before_compare_paths: Optional[List[str]] = None):
        """
        初始化比对器

        Args:
            excluded_paths: 需要排除比对的字段路径列表（JSONPath格式）
            unordered_arrays: 是否对数组进行无序比对（默认False为有序比对）
            payload_std: 标准（老系统）完整payload，用于补充差异上下文字段
            payload_new: 新系统完整payload，用于补充差异上下文字段
            unordered_array_match_keys: 无序数组键匹配配置，默认使用模块级常量
            null_equivalent_to_empty: 是否将 null 与空字符串 "" 视为等价
            trim_before_compare_paths: 比对前需要对字符串值做 trim 的路径列表
        """
        self.excluded_paths = excluded_paths or []
        self.unordered_arrays = unordered_arrays
        self.payload_std = payload_std
        self.payload_new = payload_new
        self.null_equivalent_to_empty = null_equivalent_to_empty
        self.differences = []

        # 标准化无序数组键匹配配置，方便后续用 _normalize_path(path) 直接查找
        raw_match_keys = unordered_array_match_keys or UNORDERED_ARRAY_MATCH_KEYS
        self.unordered_array_match_keys = {
            self._normalize_path(k): v for k, v in raw_match_keys.items()
        }

        # 标准化需要 trim 的路径配置
        raw_trim_paths = trim_before_compare_paths or TRIM_BEFORE_COMPARE_PATHS
        self.trim_paths = {self._normalize_path(p) for p in raw_trim_paths}
        
    def _normalize_path(self, path: str) -> str:
        """标准化路径：移除根前缀并将数组索引统一为[*]。"""
        # 示例：
        # $.a.b[0].c -> a.b[*].c
        # $.a.b[12].c -> a.b[*].c
        # 这样可以让排除路径在数组场景下稳定命中。
        clean_path = path.lstrip('$').lstrip('.')
        clean_path = re.sub(r"\[\d+\]", "[*]", clean_path)
        return clean_path

    def _is_excluded(self, path: str) -> bool:
        """
        判断路径是否在排除列表中

        Args:
            path: 字段路径

        Returns:
            是否排除该路径
        """
        normalized_path = self._normalize_path(path)
        for excluded in self.excluded_paths:
            normalized_excluded = self._normalize_path(excluded)
            if normalized_path == normalized_excluded:
                return True
        return False

    def _is_trim_path(self, path: str) -> bool:
        """
        判断当前路径是否需要在比对前对字符串值做 trim。

        Args:
            path: 字段路径

        Returns:
            是否需要 trim
        """
        return self._normalize_path(path) in self.trim_paths

    def _get_type_name(self, value: Any) -> str:
        """
        获取值的类型名称
        
        Args:
            value: 待检查的值
            
        Returns:
            类型名称字符串
        """
        if value is None:
            return "null"
        elif isinstance(value, bool):
            return "boolean"
        elif isinstance(value, int):
            return "integer"
        elif isinstance(value, float):
            return "number"
        elif isinstance(value, str):
            return "string"
        elif isinstance(value, list):
            return "array"
        elif isinstance(value, dict):
            return "object"
        else:
            return type(value).__name__
    
    def _is_null_empty_equivalent(self, value1: Any, value2: Any) -> bool:
        """
        判断 null 与严格空字符串是否应视为等价。

        Args:
            value1: 第一个值
            value2: 第二个值

        Returns:
            当一边是 None 且另一边是严格空字符串 "" 时返回 True
        """
        if not self.null_equivalent_to_empty:
            return False
        return (
            (value1 is None and isinstance(value2, str) and value2 == "")
            or (value2 is None and isinstance(value1, str) and value1 == "")
        )

    def _add_difference(self, path: str, diff_type: str,
                       value1: Any = None, value2: Any = None,
                       type1: str = None, type2: str = None,
                       extra_info: str = None):
        """
        添加差异记录

        Args:
            path: 字段路径
            diff_type: 差异类型
            value1: JSON1的值
            value2: JSON2的值
            type1: JSON1的类型
            type2: JSON2的类型
            extra_info: 额外信息
        """
        diff_record = {
            '字段路径': path,
            '差异类型': diff_type,
            'JSON1值': str(value1) if value1 is not None else '',
            'JSON2值': str(value2) if value2 is not None else '',
            'JSON1类型': type1 if type1 else '',
            'JSON2类型': type2 if type2 else '',
            '额外信息': extra_info if extra_info else ''
        }
        if self.payload_std is not None and self.payload_new is not None:
            context = _get_comparison_context(self.payload_std, self.payload_new, path)
            diff_record.update(context)
        self.differences.append(diff_record)
    
    def compare(self, json1: Any, json2: Any, path: str = "$") -> List[Dict]:
        """
        递归比对两个JSON对象
        
        Args:
            json1: 第一个JSON对象
            json2: 第二个JSON对象
            path: 当前字段路径
            
        Returns:
            差异列表
        """
        # 检查是否为排除字段
        if self._is_excluded(path):
            return self.differences
        
        type1 = self._get_type_name(json1)
        type2 = self._get_type_name(json2)
        
        # 类型不一致
        if type1 != type2:
            # null 与空字符串视为等价，减少系统间空值表示差异导致的误报
            if self._is_null_empty_equivalent(json1, json2):
                return self.differences
            self._add_difference(
                path=path,
                diff_type='类型不匹配',
                value1=json1,
                value2=json2,
                type1=type1,
                type2=type2
            )
            return self.differences
        
        # 处理对象类型
        if isinstance(json1, dict) and isinstance(json2, dict):
            self._compare_objects(json1, json2, path)
        
        # 处理数组类型
        elif isinstance(json1, list) and isinstance(json2, list):
            self._compare_arrays(json1, json2, path)
        
        # 处理基本类型
        else:
            comparable1, comparable2 = json1, json2
            if (self._is_trim_path(path)
                    and isinstance(json1, str)
                    and isinstance(json2, str)):
                comparable1, comparable2 = json1.strip(), json2.strip()

            if comparable1 != comparable2:
                self._add_difference(
                    path=path,
                    diff_type='值不相等',
                    value1=json1,
                    value2=json2,
                    type1=type1,
                    type2=type2
                )
        
        return self.differences
    
    def _compare_objects(self, obj1: dict, obj2: dict, path: str):
        """
        比对两个对象
        
        Args:
            obj1: 第一个对象
            obj2: 第二个对象
            path: 当前路径
        """
        keys1 = set(obj1.keys())
        keys2 = set(obj2.keys())
        
        # 检查缺失的字段
        only_in_json1 = keys1 - keys2
        only_in_json2 = keys2 - keys1
        common_keys = keys1 & keys2
        
        for key in only_in_json1:
            field_path = f"{path}.{key}"
            if not self._is_excluded(field_path):
                self._add_difference(
                    path=field_path,
                    diff_type='字段仅存在于JSON1',
                    value1=obj1[key],
                    type1=self._get_type_name(obj1[key]),
                    extra_info=f'JSON2缺少此字段'
                )
        
        for key in only_in_json2:
            field_path = f"{path}.{key}"
            if not self._is_excluded(field_path):
                self._add_difference(
                    path=field_path,
                    diff_type='字段仅存在于JSON2',
                    value2=obj2[key],
                    type2=self._get_type_name(obj2[key]),
                    extra_info=f'JSON1缺少此字段'
                )
        
        # 递归比对共同字段
        for key in common_keys:
            field_path = f"{path}.{key}"
            self.compare(obj1[key], obj2[key], field_path)
    
    def _compare_arrays(self, arr1: list, arr2: list, path: str):
        """
        比对两个数组
        
        Args:
            arr1: 第一个数组
            arr2: 第二个数组
            path: 当前路径
        """
        if self.unordered_arrays:
            self._compare_arrays_unordered(arr1, arr2, path)
        else:
            self._compare_arrays_ordered(arr1, arr2, path)
    
    def _compare_arrays_ordered(self, arr1: list, arr2: list, path: str):
        """
        有序比对两个数组
        
        Args:
            arr1: 第一个数组
            arr2: 第二个数组
            path: 当前路径
        """
        len1 = len(arr1)
        len2 = len(arr2)
        
        # 数组长度不一致
        if len1 != len2:
            self._add_difference(
                path=path,
                diff_type='数组长度不一致',
                value1=f'长度: {len1}',
                value2=f'长度: {len2}',
                type1='array',
                type2='array',
                extra_info=f'JSON1有{len1}个元素，JSON2有{len2}个元素'
            )
        
        # 逐个比对数组元素
        min_len = min(len1, len2)
        for i in range(min_len):
            element_path = f"{path}[{i}]"
            self.compare(arr1[i], arr2[i], element_path)
        
        # 处理长度不一致时多出的元素
        if len1 > len2:
            for i in range(len2, len1):
                element_path = f"{path}[{i}]"
                if not self._is_excluded(element_path):
                    self._add_difference(
                        path=element_path,
                        diff_type='元素仅存在于JSON1',
                        value1=arr1[i],
                        type1=self._get_type_name(arr1[i]),
                        extra_info=f'JSON2的数组较短'
                    )
        elif len2 > len1:
            for i in range(len1, len2):
                element_path = f"{path}[{i}]"
                if not self._is_excluded(element_path):
                    self._add_difference(
                        path=element_path,
                        diff_type='元素仅存在于JSON2',
                        value2=arr2[i],
                        type2=self._get_type_name(arr2[i]),
                        extra_info=f'JSON1的数组较短'
                    )
    
    def _get_unordered_array_match_key(
        self,
        element: Any,
        key_fields: List[str],
        preserve_null_fields: List[str]
    ) -> Optional[tuple]:
        """
        从数组元素中提取无序数组匹配键。

        Args:
            element: 数组元素
            key_fields: 组成匹配键的字段路径列表（相对于数组元素本身）
            preserve_null_fields: 需要保留 None 的字段路径列表

        Returns:
            匹配键元组；元素非 dict 或无法提取任何键字段时返回 None
        """
        if not isinstance(element, dict):
            return None

        key_values = []
        has_valid_field = False
        for field_path in key_fields:
            parts = field_path.split(".")
            value = element
            for part in parts:
                if isinstance(value, dict) and part in value:
                    value = value[part]
                else:
                    value = None
                    break

            if value is not None:
                has_valid_field = True

            if field_path in preserve_null_fields:
                key_values.append(value)  # 保留 None，null 只匹配 null
            else:
                key_values.append(_norm_str(value))  # None/NaN -> ""

        return tuple(key_values) if has_valid_field else None

    def _compare_arrays_unordered(self, arr1: list, arr2: list, path: str):
        """
        无序比对两个数组（通过内容匹配）

        对于配置了匹配键的数组（如 DerivedBestRecord[*]），按指定字段组合直接查找；
        其他数组仍使用差异最小的贪心策略。

        Args:
            arr1: 第一个数组
            arr2: 第二个数组
            path: 当前路径
        """
        len1 = len(arr1)
        len2 = len(arr2)

        # 数组长度不一致
        if len1 != len2:
            self._add_difference(
                path=path,
                diff_type='数组长度不一致',
                value1=f'长度: {len1}',
                value2=f'长度: {len2}',
                type1='array',
                type2='array',
                extra_info=f'JSON1有{len1}个元素，JSON2有{len2}个元素（无序比对）'
            )

        # 创建副本用于匹配
        arr2_remaining = list(range(len2))
        matched_arr2 = set()
        match_plan: List[tuple] = []  # (i_from_arr1, j_from_arr2)

        normalized_path = self._normalize_path(path)
        match_config = self.unordered_array_match_keys.get(normalized_path)

        if match_config:
            # 按配置字段组合进行键匹配
            key_fields = match_config["fields"]
            preserve_null = match_config.get("preserve_null", [])

            # 为 arr2 建立匹配键索引：key -> list of indices
            arr2_key_index: Dict[tuple, List[int]] = {}
            for j in arr2_remaining:
                key = self._get_unordered_array_match_key(arr2[j], key_fields, preserve_null)
                if key is not None:
                    arr2_key_index.setdefault(key, []).append(j)

            # 为 arr1 中每个元素按键查找匹配
            for i, item1 in enumerate(arr1):
                key = self._get_unordered_array_match_key(item1, key_fields, preserve_null)
                if key is not None and arr2_key_index.get(key):
                    j = arr2_key_index[key].pop(0)
                    matched_arr2.add(j)
                    match_plan.append((i, j))
                # 找不到匹配时暂不处理，最后统一记录未匹配差异

            arr2_remaining = [j for j in arr2_remaining if j not in matched_arr2]

            # 为未匹配的 arr1 元素记录差异
            matched_arr1_indices = {pair[0] for pair in match_plan}
            for i, item1 in enumerate(arr1):
                if i in matched_arr1_indices:
                    continue
                element_path = f"{path}[{i}]"
                if not self._is_excluded(element_path):
                    self._add_difference(
                        path=element_path,
                        diff_type='数组元素无匹配（无序）',
                        value1=item1,
                        type1=self._get_type_name(item1),
                        extra_info='按匹配键未找到对应元素'
                    )

            # 执行已配对元素的比对
            for i, j in match_plan:
                element_path = f"{path}[{i}]"
                self.compare(arr1[i], arr2[j], element_path)

        else:
            # 未配置匹配键：使用差异最小的贪心策略
            for i, item1 in enumerate(arr1):
                best_match_idx = None
                best_match_diff_count = float('inf')

                for j in arr2_remaining:
                    item2 = arr2[j]
                    # 创建临时比对器来计算差异数
                    temp_comparator = JsonComparator(
                        excluded_paths=self.excluded_paths,
                        unordered_arrays=self.unordered_arrays,
                        unordered_array_match_keys=self.unordered_array_match_keys,
                        null_equivalent_to_empty=self.null_equivalent_to_empty,
                    )
                    temp_comparator.compare(item1, item2, "$temp")
                    diff_count = len(temp_comparator.differences)

                    # 找到完全匹配或最佳匹配
                    if diff_count == 0:
                        best_match_idx = j
                        break
                    elif diff_count < best_match_diff_count:
                        best_match_diff_count = diff_count
                        best_match_idx = j

                if best_match_idx is not None:
                    arr2_remaining.remove(best_match_idx)
                    element_path = f"{path}[{i}]"
                    self.compare(arr1[i], arr2[best_match_idx], element_path)
                else:
                    # arr1中的元素在arr2中找不到匹配
                    element_path = f"{path}[{i}]"
                    if not self._is_excluded(element_path):
                        self._add_difference(
                            path=element_path,
                            diff_type='数组元素无匹配（无序）',
                            value1=arr1[i],
                            type1=self._get_type_name(arr1[i]),
                            extra_info=f'在JSON2中找不到匹配的元素'
                        )

        # 处理arr2中未匹配的元素
        for j in arr2_remaining:
            element_path = f"{path}[{j}]"
            if not self._is_excluded(element_path):
                self._add_difference(
                    path=element_path,
                    diff_type='数组元素无匹配（无序）',
                    value2=arr2[j],
                    type2=self._get_type_name(arr2[j]),
                    extra_info=f'在JSON1中找不到匹配的元素'
                )

    def generate_report(self, output_file: str):
        """
        生成Excel格式的比对报告

        Args:
            output_file: 输出文件路径
        """
        if not self.differences:
            # 没有差异，创建一个说明
            df = pd.DataFrame([{
                '比对结果': '两个JSON完全一致（排除字段除外）',
                '比对时间': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            }])
        else:
            # 有差异，按字段路径排序
            df = pd.DataFrame(self.differences)
            df = df.sort_values(by='字段路径')
            # 确保报告列顺序一致，缺失列补空
            for col in DIFF_REPORT_COLUMNS:
                if col not in df.columns:
                    df[col] = ''
            df = df[DIFF_REPORT_COLUMNS]

        # 创建Excel写入器
        with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
            # 写入差异明细
            df.to_excel(writer, sheet_name='差异明细', index=False)

            # 写入统计摘要
            if self.differences:
                summary_data = {
                    '统计项': ['总差异数', '类型不匹配', '值不相等', '字段缺失', '数组长度不一致', '数组元素差异'],
                    '数量': [
                        len(self.differences),
                        len([d for d in self.differences if d['差异类型'] == '类型不匹配']),
                        len([d for d in self.differences if d['差异类型'] == '值不相等']),
                        len([d for d in self.differences if '字段仅存在' in d['差异类型']]),
                        len([d for d in self.differences if d['差异类型'] == '数组长度不一致']),
                        len([d for d in self.differences if '元素仅存在' in d['差异类型']])
                    ]
                }
                summary_df = pd.DataFrame(summary_data)
                summary_df.to_excel(writer, sheet_name='统计摘要', index=False)

        print(f"比对报告已生成: {output_file}")
        print(f"发现 {len(self.differences)} 处差异")
    
    def generate_html_report(self, output_file: str):
        """
        生成HTML格式的比对报告
        
        Args:
            output_file: 输出文件路径
        """
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        
        html_content = f"""
<!DOCTYPE html>
<html lang="zh-CN">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>JSON比对报告</title>
    <style>
        body {{
            font-family: 'Microsoft YaHei', Arial, sans-serif;
            margin: 0;
            padding: 20px;
            background-color: #f5f5f5;
        }}
        .container {{
            max-width: 1400px;
            margin: 0 auto;
            background-color: white;
            padding: 30px;
            border-radius: 8px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.1);
        }}
        h1 {{
            color: #2c3e50;
            border-bottom: 3px solid #3498db;
            padding-bottom: 10px;
        }}
        h2 {{
            color: #34495e;
            margin-top: 30px;
            border-left: 4px solid #3498db;
            padding-left: 15px;
        }}
        .summary {{
            background-color: #ecf0f1;
            padding: 20px;
            border-radius: 5px;
            margin: 20px 0;
        }}
        .summary-item {{
            display: inline-block;
            margin: 10px 20px 10px 0;
            font-size: 16px;
        }}
        .summary-label {{
            font-weight: bold;
            color: #7f8c8d;
        }}
        .summary-value {{
            color: #e74c3c;
            font-weight: bold;
            font-size: 18px;
        }}
        table {{
            width: 100%;
            border-collapse: collapse;
            margin: 20px 0;
            box-shadow: 0 1px 3px rgba(0,0,0,0.1);
        }}
        th {{
            background-color: #3498db;
            color: white;
            padding: 12px;
            text-align: left;
            font-weight: bold;
        }}
        td {{
            padding: 10px 12px;
            border-bottom: 1px solid #ddd;
        }}
        tr:hover {{
            background-color: #f8f9fa;
        }}
        .diff-type {{
            padding: 4px 8px;
            border-radius: 3px;
            font-size: 12px;
            font-weight: bold;
        }}
        .type-mismatch {{
            background-color: #ffebee;
            color: #c62828;
        }}
        .value-diff {{
            background-color: #fff3e0;
            color: #e65100;
        }}
        .field-missing {{
            background-color: #f3e5f5;
            color: #6a1b9a;
        }}
        .array-diff {{
            background-color: #e8f5e9;
            color: #2e7d32;
        }}
        .path {{
            font-family: 'Courier New', monospace;
            background-color: #f8f9fa;
            padding: 2px 6px;
            border-radius: 3px;
            font-size: 13px;
        }}
        .value {{
            font-family: 'Courier New', monospace;
            font-size: 12px;
            max-width: 300px;
            overflow: hidden;
            text-overflow: ellipsis;
            white-space: nowrap;
        }}
        .no-diff {{
            text-align: center;
            padding: 40px;
            color: #27ae60;
            font-size: 18px;
            font-weight: bold;
        }}
        .footer {{
            margin-top: 30px;
            text-align: center;
            color: #95a5a6;
            font-size: 14px;
        }}
    </style>
</head>
<body>
    <div class="container">
        <h1>📊 JSON比对报告</h1>
        <div class="summary">
            <div class="summary-item">
                <span class="summary-label">比对时间：</span>
                <span>{timestamp}</span>
            </div>
            <div class="summary-item">
                <span class="summary-label">比对模式：</span>
                <span>{'无序数组比对' if self.unordered_arrays else '有序数组比对'}</span>
            </div>
            <div class="summary-item">
                <span class="summary-label">总差异数：</span>
                <span class="summary-value">{len(self.differences)}</span>
            </div>
        </div>
"""
        
        if not self.differences:
            html_content += """
        <div class="no-diff">
            ✅ 两个JSON完全一致（排除字段除外）
        </div>
"""
        else:
            # 统计摘要
            stats = {
                '类型不匹配': len([d for d in self.differences if d['差异类型'] == '类型不匹配']),
                '值不相等': len([d for d in self.differences if d['差异类型'] == '值不相等']),
                '字段缺失': len([d for d in self.differences if '字段仅存在' in d['差异类型']]),
                '数组差异': len([d for d in self.differences if '数组' in d['差异类型']])
            }
            
            html_content += """
        <h2>📈 统计摘要</h2>
        <table>
            <tr>
                <th>差异类型</th>
                <th>数量</th>
            </tr>
"""
            for stat_name, stat_count in stats.items():
                html_content += f"""
            <tr>
                <td>{stat_name}</td>
                <td>{stat_count}</td>
            </tr>
"""
            
            html_content += """
        </table>
        
        <h2>📋 差异明细</h2>
        <table>
            <tr>
                <th style="width: 25%;">字段路径</th>
                <th style="width: %;">差异类型</th>
                <th style="width: 15%;">JSON1值</th>
                <th style="width: 15%;">JSON2值</th>
                <th style="width: 10%;">JSON1类型</th>
                <th style="width: 10%;">JSON2类型</th>
                <th style="width: 20%;">额外信息</th>
            </tr>
"""
            
            for diff in sorted(self.differences, key=lambda x: x['字段路径']):
                # 确定差异类型的CSS类
                diff_class = 'diff-type'
                if '类型不匹配' in diff['差异类型']:
                    diff_class += ' type-mismatch'
                elif '值不相等' in diff['差异类型']:
                    diff_class += ' value-diff'
                elif '字段仅存在' in diff['差异类型']:
                    diff_class += ' field-missing'
                else:
                    diff_class += ' array-diff'
                
                html_content += f"""
            <tr>
                <td><span class="path">{self._html_escape(diff['字段路径'])}</span></td>
                <td><span class="{diff_class}">{self._html_escape(diff['差异类型'])}</span></td>
                <td><span class="value">{self._html_escape(diff['JSON1值'][:100]) if diff['JSON1值'] else '-'}</span></td>
                <td><span class="value">{self._html_escape(diff['JSON2值'][:100]) if diff['JSON2值'] else '-'}</span></td>
                <td>{self._html_escape(diff['JSON1类型']) if diff['JSON1类型'] else '-'}</td>
                <td>{self._html_escape(diff['JSON2类型']) if diff['JSON2类型'] else '-'}</td>
                <td>{self._html_escape(diff['额外信息']) if diff['额外信息'] else '-'}</td>
            </tr>
"""
            
            html_content += """
        </table>
"""
        
        html_content += """
        <div class="footer">
            <p>生成时间: {timestamp}</p>
            <p>JSON比对工具 v2.0 - 支持JSON字符串比对、无序数组比对、HTML报告生成</p>
        </div>
    </div>
</body>
</html>
""".format(timestamp=timestamp)
        
        # 写入HTML文件
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(html_content)
        
        print(f"HTML报告已生成: {output_file}")
    
    def _html_escape(self, text: str) -> str:
        """
        HTML转义
        
        Args:
            text: 原始文本
            
        Returns:
            转义后的文本
        """
        if not isinstance(text, str):
            text = str(text)
        return (text.replace('&', '&amp;')
                   .replace('<', '&lt;')
                   .replace('>', '&gt;')
                   .replace('"', '&quot;')
                   .replace("'", '&#39;'))


def _norm_str(value: Any) -> str:
    """将任意输入规范化为可比较字符串。"""
    # Excel读出的空值常是NaN，这里统一转成空字符串，
    # 避免后续cid/uid判断出现“看似有值”的误判。
    if value is None:
        return ""
    if isinstance(value, float) and pd.isna(value):
        return ""
    return str(value).strip()


def _is_non_empty(value: Any) -> bool:
    """判断值是否为有效非空内容。"""
    return _norm_str(value) != ""


def _extract_values_by_path(payload: Dict[str, Any], json_path: str) -> List[Any]:
    """按简化JSONPath提取值（支持对象层级、[*]数组通配与[数字]具体索引）。"""
    parts = [p for p in json_path.lstrip("$").lstrip(".").split(".") if p]
    current_values: List[Any] = [payload]

    for part in parts:
        next_values: List[Any] = []
        # 支持具体索引 [n] 和通配 [*]
        index_match = re.match(r"^(.+)\[(\d+)\]$", part)
        is_array_wildcard = part.endswith("[*]")

        if index_match:
            key = index_match.group(1)
            index = int(index_match.group(2))
        elif is_array_wildcard:
            key = part[:-3]
            index = None
        else:
            key = part
            index = None

        for current in current_values:
            if not isinstance(current, dict) or key not in current:
                continue

            value = current[key]
            if is_array_wildcard:
                if isinstance(value, list):
                    next_values.extend(value)
                elif value is not None:
                    next_values.append(value)
            elif index_match is not None:
                if isinstance(value, list) and 0 <= index < len(value):
                    next_values.append(value[index])
                elif value is not None:
                    next_values.append(value)
            else:
                next_values.append(value)

        current_values = next_values
        if not current_values:
            return []

    return current_values


def extract_value_by_path(payload: Dict[str, Any], json_path: str) -> Any:
    """提取首个值，兼容旧调用。"""
    values = _extract_values_by_path(payload, json_path)
    return values[0] if values else None


def _extract_non_empty_str_values(payload: Dict[str, Any], json_path: str) -> List[str]:
    """提取路径对应的非空字符串值并去重。"""
    values: List[str] = []
    seen = set()
    for raw in _extract_values_by_path(payload, json_path):
        v = _norm_str(raw)
        if not v or v in seen:
            continue
        seen.add(v)
        values.append(v)
    return values


def _unwrap_single_element_arrays(obj: Any) -> Any:
    """递归展开长度为1的数组，使单元素数组归一化为数组内的唯一元素。"""
    if isinstance(obj, dict):
        return {k: _unwrap_single_element_arrays(v) for k, v in obj.items()}
    if isinstance(obj, list):
        if len(obj) == 1:
            return _unwrap_single_element_arrays(obj[0])
        return [_unwrap_single_element_arrays(item) for item in obj]
    return obj


def _join_values(values: List[str]) -> str:
    """将值列表用逗号连接并去重，保持原有顺序。"""
    seen = set()
    result: List[str] = []
    for v in values:
        if v and v not in seen:
            seen.add(v)
            result.append(v)
    return ", ".join(result)


def _extract_cid_list_for_context(payload: Dict[str, Any], derived_index: Optional[int] = None) -> str:
    """根据上下文提取 CID List。

    Args:
        payload: 消息体
        derived_index: None 表示 BestRecord 上下文；int 表示 DerivedBestRecord 的索引
    """
    if derived_index is None:
        json_path = "$.ConsumerBestRecord.BestRecord.SourceSystemList.SourceSystem[*].ConsumerId"
    else:
        json_path = (
            f"$.ConsumerBestRecord.DerivedBestRecordList.DerivedBestRecord[{derived_index}]"
            ".SourceSystemList.SourceSystem[*].ConsumerId"
        )
    values = _extract_non_empty_str_values(payload, json_path)
    return _join_values(values)


def _get_comparison_context(
    payload_std: Dict[str, Any], payload_new: Dict[str, Any], diff_path: str
) -> Dict[str, str]:
    """根据差异路径获取上下文字段。

    返回字段：uid1, uid1-L1-RecordTimestamp, uid2, uid2-L1-RecordTimestamp,
            Market, Brand, DistributionChannelCode, CID List 1, CID List 2
    """
    # 全局字段
    uid1 = extract_value_by_path(payload_std, UID_JSON_PATH)
    uid1_ts = extract_value_by_path(payload_std, "$.ConsumerBestRecord.BestRecord.Attributes.RecordTimestamp")
    uid2 = extract_value_by_path(payload_new, UID_JSON_PATH)
    uid2_ts = extract_value_by_path(payload_new, "$.ConsumerBestRecord.BestRecord.Attributes.RecordTimestamp")
    market = extract_value_by_path(payload_std, "$.ConsumerBestRecord.BestRecord.PersonalData.RegTouchPointSourceSystem.MarketCode")
    # market = extract_value_by_path(payload_std, "$.ConsumerBestRecord.BestRecord.SourceSystemList.SourceSystem[1].MarketCode")

    # 判断差异位于 BestRecord 还是 DerivedBestRecordList.DerivedBestRecord[i]
    derived_match = re.match(
        r"^\$\.ConsumerBestRecord\.DerivedBestRecordList\.DerivedBestRecord\[(\d+)\]",
        diff_path,
    )
    if derived_match:
        idx = int(derived_match.group(1))
        brand = extract_value_by_path(
            payload_std,
            f"$.ConsumerBestRecord.DerivedBestRecordList.DerivedBestRecord[{idx}].Attributes.BrandCode",
        )
        dist = extract_value_by_path(
            payload_std,
            f"$.ConsumerBestRecord.DerivedBestRecordList.DerivedBestRecord[{idx}].Attributes.DistributionChannelCode",
        )
        cid_list_1 = _extract_cid_list_for_context(payload_std, derived_index=idx)
        cid_list_2 = _extract_cid_list_for_context(payload_new, derived_index=idx)
    else:
        brand = None
        dist = None
        cid_list_1 = _extract_cid_list_for_context(payload_std, derived_index=None)
        cid_list_2 = _extract_cid_list_for_context(payload_new, derived_index=None)

    return {
        "uid1": _norm_str(uid1),
        "uid1-L1-RecordTimestamp": _norm_str(uid1_ts),
        "uid2": _norm_str(uid2),
        "uid2-L1-RecordTimestamp": _norm_str(uid2_ts),
        "Market": _norm_str(market),
        "Brand": _norm_str(brand),
        "DistributionChannelCode": _norm_str(dist),
        "CID List 1": cid_list_1,
        "CID List 2": cid_list_2,
    }


def load_rules(config_path: Path) -> List[Dict[str, str]]:
    """读取配置规则并做基础结构校验。"""
    # 仅做列级校验，不在这里过滤空规则。
    # 空规则保留给主流程统一记录为SKIPPED_EMPTY_RULE，
    # 方便报告中完整追踪配置质量。
    if not config_path.exists():
        raise FileNotFoundError(f"Config not found: {config_path}")

    df = pd.read_excel(config_path)
    if df.empty:
        raise Exception("rule config is empty.")
    required_cols = {"input_cid1", "input_cid2", "input_uid1", "input_uid2"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Config missing columns: {', '.join(sorted(missing))}")

    rules: List[Dict[str, str]] = []
    for idx, row in df.iterrows():
        input_cid1 = _norm_str(row.get("input_cid1"))
        input_cid2 = _norm_str(row.get("input_cid2"))
        input_uid1 = _norm_str(row.get("input_uid1"))
        input_uid2 = _norm_str(row.get("input_uid2"))
        rules.append({
            "rule_id": str(idx + 1),
            "input_cid1": input_cid1,
            "input_cid2": input_cid2,
            "input_uid1": input_uid1,
            "input_uid2": input_uid2,
        })

    valid_count = sum(
        1 for r in rules
        if (
            (_is_non_empty(r["input_uid1"]) or _is_non_empty(r["input_cid1"]))
            and (_is_non_empty(r["input_uid2"]) or _is_non_empty(r["input_cid2"]))
        )
    )
    skipped_count = len(rules) - valid_count
    logger.info(
        "配置加载完成: total_rules=%s, valid_rules=%s, empty_rules=%s, config=%s",
        len(rules),
        valid_count,
        skipped_count,
        config_path,
    )
    return rules


def _key_deserializer(k: bytes) -> Optional[str]:
    """将 Kafka message key 解码为 UID 字符串；空 key 返回 None。"""
    if k is None:
        return None
    text = k.decode("utf-8-sig", errors="replace").strip()
    return text if text else None


def _parse_payload_bytes(v: bytes) -> Optional[Dict[str, Any]]:
    """安全反序列化 Kafka 消息 value（bytes），仅返回 JSON 对象。"""
    if v is None:
        return None
    text = v.decode("utf-8-sig", errors="replace").strip()
    if not text:
        return None
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError as exc:
        sample = text[:200].replace("\n", "\\n")
        logger.warning(
            "Kafka消息非标准JSON，已跳过: %s: %s, sample=%s",
            type(exc).__name__,
            exc,
            sample,
        )
        return None
    if not isinstance(parsed, dict):
        logger.warning("Kafka消息非JSON对象，已跳过: type=%s", type(parsed).__name__)
        return None
    return parsed


def _build_consumer(topics: List[str]):
    """为多个topic创建KafkaConsumer（key 反序列化为 UID，value 保留原始 bytes）。"""
    try:
        from kafka import KafkaConsumer  # type: ignore
    except ImportError as exc:
        raise ImportError("缺少依赖 kafka-python，请先安装: pip install kafka-python") from exc

    logger.debug("初始化Kafka consumer: topics=%s, bootstrap=%s", topics, KAFKA_BOOTSTRAP_SERVERS)

    return KafkaConsumer(
        *topics,
        bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
        auto_offset_reset="earliest",
        enable_auto_commit=False,
        consumer_timeout_ms=KAFKA_CONSUMER_TIMEOUT_MS,
        key_deserializer=_key_deserializer,
        # value 保持原始 bytes，命中后再解析，减少扫描阶段开销
    )


def fetch_available_topics() -> Optional[set]:
    """从Kafka拉取当前可用topic集合。"""
    try:
        from kafka import KafkaConsumer  # type: ignore
    except ImportError as exc:
        raise ImportError("缺少依赖 kafka-python，请先安装: pip install kafka-python") from exc

    consumer = KafkaConsumer(
        bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
        enable_auto_commit=False,
        consumer_timeout_ms=KAFKA_CONSUMER_TIMEOUT_MS,
    )
    try:
        topics = consumer.topics()
        logger.info("topic预检查完成: available_topics=%s", len(topics))
        return topics
    except Exception as exc:
        logger.warning("topic预检查失败，将跳过存在性预过滤: %s: %s", type(exc).__name__, exc)
        return None
    finally:
        consumer.close()


def _update_latest(bucket: Dict[Any, Dict[str, Any]], key: Any, record: Dict[str, Any]) -> None:
    """同一key只保留timestamp最新的记录。"""
    current = bucket.get(key)
    if current is None or record["timestamp"] > current["timestamp"]:
        bucket[key] = record

def scan_latest_messages_for_topics(topics: List[str], group_name: str) -> Dict[str, Dict[str, Dict[Any, Dict[str, Any]]]]:
    """一次扫描一组topic，为后续规则匹配构建 UID 索引。"""
    # 返回结构：
    # {
    #   topic: {
    #     "by_uid": {uid: latest_record},
    #   }
    # }
    if not topics:
        logger.info("跳过扫描%s: 无可用topic", group_name)
        return {}

    logger.info(
        "开始批量扫描%s: topic_count=%s",
        group_name,
        len(topics),
    )

    consumer = _build_consumer(topics)
    indexes_by_topic: Dict[str, Dict[str, Dict[Any, Dict[str, Any]]]] = {
        t: {"by_uid": {}} for t in topics
    }
    scanned_records = 0
    indexed_records = 0

    try:
        # 连续多次poll无数据后退出，避免长时间阻塞。
        empty_polls = 0
        while empty_polls < KAFKA_MAX_EMPTY_POLLS:
            polled = consumer.poll(timeout_ms=KAFKA_POLL_TIMEOUT_MS)
            if not polled:
                empty_polls += 1
                continue

            empty_polls = 0
            for _tp, records in polled.items():
                for rec in records:
                    scanned_records += 1
                    # 直接用 Kafka message key 作为 UID；value 保留 bytes 延迟解析。
                    uid_val = rec.key
                    raw_value = rec.value
                    if uid_val is None or raw_value is None:
                        continue

                    latest_record = {
                        "topic": rec.topic,
                        "timestamp": rec.timestamp,
                        "offset": rec.offset,
                        "payload": raw_value,
                    }

                    topic_index = indexes_by_topic.setdefault(rec.topic, {"by_uid": {}})

                    _update_latest(topic_index["by_uid"], uid_val, latest_record)
                    indexed_records += 1
    finally:
        consumer.close()

    logger.info(
        "%s扫描结束: topic_count=%s, scanned=%s, indexed=%s",
        group_name,
        len(topics),
        scanned_records,
        indexed_records,
    )

    return indexes_by_topic


def get_latest_message_from_index(topic_index: Optional[Dict[str, Dict[Any, Dict[str, Any]]]], input_uid: str) -> Optional[Dict[str, Any]]:
    """按 input_uid 从topic索引中提取最新命中消息。

    Args:
        topic_index: 某个topic的索引，结构为 {"by_uid": {uid: latest_record}}。
        input_uid: 规则中输入的 UID。

    Returns:
        命中的最新记录；input_uid 为空或索引不存在时返回 None。
    """
    if topic_index is None:
        return None

    if _is_non_empty(input_uid):
        return topic_index["by_uid"].get(input_uid)
    return None


def compare_payloads(payload_std: Dict[str, Any], payload_new: Dict[str, Any], excluded_paths: List[str],
                     normalize_new_single_element_arrays: bool = True,
                     null_equivalent_to_empty: bool = NULL_EQUIVALENT_TO_EMPTY_STRING,
                     trim_before_compare_paths: Optional[List[str]] = None) -> List[Dict[str, str]]:
    """比对两个payload并返回差异列表。"""
    # 使用无序数组比对，避免数组顺序波动导致误报。
    if normalize_new_single_element_arrays:
        payload_new = _unwrap_single_element_arrays(payload_new)
    comparator = JsonComparator(
        excluded_paths=excluded_paths,
        unordered_arrays=True,
        payload_std=payload_std,
        payload_new=payload_new,
        null_equivalent_to_empty=null_equivalent_to_empty,
        trim_before_compare_paths=trim_before_compare_paths,
    )
    comparator.compare(payload_std, payload_new)
    return comparator.differences

def generate_batch_report(summary_rows: List[Dict[str, Any]], diff_rows: List[Dict[str, Any]], output_file: Path) -> None:
    """生成批量执行报告。"""
    # 任务汇总：每条规则在每个topic映射上的执行状态
    # 差异明细：仅记录确实产生的字段级差异或缺失类差异
    summary_df = pd.DataFrame(summary_rows)
    diff_df = pd.DataFrame(diff_rows)

    # 统一差异明细列顺序，确保上下文信息（uid1/Market/CID List等）按固定顺序输出
    if not diff_df.empty:
        for col in BATCH_DIFF_REPORT_COLUMNS:
            if col not in diff_df.columns:
                diff_df[col] = ''
        other_cols = [c for c in diff_df.columns if c not in BATCH_DIFF_REPORT_COLUMNS]
        diff_df = diff_df[BATCH_DIFF_REPORT_COLUMNS + other_cols]

    import tempfile
    # Databricks部分挂载文件系统不支持openpyxl写入过程中需要的随机seek操作，
    # 这里先写到本地/tmp，再用dbutils.fs.cp复制到目标路径。
    with tempfile.NamedTemporaryFile(suffix=".xlsx", delete=False) as tmp:
        tmp_path = Path(tmp.name)

    def _to_dbfs_uri(path_obj: Path) -> str:
        path_str = str(path_obj)
        if path_str.startswith("dbfs:/"):
            return path_str
        if path_str.startswith("/dbfs/"):
            return "dbfs:/" + path_str[len("/dbfs/"):]
        # Databricks中常见绝对路径也按dbfs处理
        if path_str.startswith("/"):
            return "dbfs:" + path_str
        # 相对路径兜底：按dbfs根目录处理
        return "dbfs:/" + path_str

    try:
        with pd.ExcelWriter(tmp_path, engine="openpyxl") as writer:
            if not summary_df.empty:
                summary_df.to_excel(writer, sheet_name="任务汇总", index=False)
            else:
                pd.DataFrame([{"信息": "无任务数据"}]).to_excel(writer, sheet_name="任务汇总", index=False)

            if not diff_df.empty:
                diff_df.to_excel(writer, sheet_name="差异明细", index=False)
            else:
                pd.DataFrame([{"信息": "无差异"}]).to_excel(writer, sheet_name="差异明细", index=False)

        target_uri = _to_dbfs_uri(output_file)
        target_dir_uri = target_uri.rsplit("/", 1)[0]
        dbutils.fs.mkdirs(target_dir_uri)
        dbutils.fs.cp(f"file:{tmp_path}", target_uri, True)
    finally:
        if tmp_path.exists():
            tmp_path.unlink()

    logger.info(
        "报告写入完成: report=%s, summary_rows=%s, diff_rows=%s",
        output_file,
        len(summary_rows),
        len(diff_rows),
    )

def main():
    """主函数：按配置逐条规则执行topic映射比对。"""
    # 批处理主流程：
    # 1) 读取规则
    # 2) 逐规则、逐topic映射执行检索和比对
    # 3) 汇总所有结果到一个批量报告
    rules = load_rules(CONFIG_PATH)
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    logger.info("开始批量比对: topic_pairs=%s, report_dir=%s", len(topic_mapping), REPORT_DIR)

    available_topics = fetch_available_topics()

    if not rules:
        logger.warning("未读取到任何配置规则，跳过执行且不生成报告。")
        return

    valid_rules = [
        r for r in rules
        if (
            (_is_non_empty(r.get("input_uid1")) or _is_non_empty(r.get("input_cid1")))
            and (_is_non_empty(r.get("input_uid2")) or _is_non_empty(r.get("input_cid2")))
        )
    ]
    if not valid_rules:
        logger.warning("配置规则存在但两侧均无可输入项，跳过执行且不生成报告。")
        return
    logger.info("有效规则检查通过: valid_rules=%s, total_rules=%s", len(valid_rules), len(rules))

    # 全局topic集合（后续所有规则共享）
    standard_topics = list(topic_mapping.keys())
    new_topics = list(dict.fromkeys(topic_mapping.values()))

    if available_topics is not None:
        available_standard_topics = [t for t in standard_topics if t in available_topics]
        available_new_topics = [t for t in new_topics if t in available_topics]
    else:
        available_standard_topics = standard_topics
        available_new_topics = new_topics

    if not available_standard_topics:
        logger.warning("所有标准topic均不存在，结束执行。")
        return

    # 全局仅扫描两次：标准topic组一次 + 新topic组一次
    std_indexes = scan_latest_messages_for_topics(available_standard_topics, "标准topic组")
    new_indexes = scan_latest_messages_for_topics(available_new_topics, "新topic组")

    summary_rows: List[Dict[str, Any]] = []
    diff_rows: List[Dict[str, Any]] = []

    for rule in rules:
        rule_id = rule["rule_id"]
        input_cid1 = rule["input_cid1"]
        input_cid2 = rule["input_cid2"]
        input_uid1 = rule["input_uid1"]
        input_uid2 = rule["input_uid2"]
        rule_topic_count = 0

        # 规则行保护：两侧均无有效输入时跳过并写入报告
        std_has_input = _is_non_empty(input_uid1) or _is_non_empty(input_cid1)
        new_has_input = _is_non_empty(input_uid2) or _is_non_empty(input_cid2)
        if not std_has_input and not new_has_input:
            logger.warning("规则跳过: rule_id=%s, reason=all_inputs_empty", rule_id)
            summary_rows.append(
                {
                    "rule_id": rule_id,
                    "input_cid1": input_cid1,
                    "input_cid2": input_cid2,
                    "input_uid1": input_uid1,
                    "input_uid2": input_uid2,
                    "standard_topic": "",
                    "new_topic": "",
                    "status": "SKIPPED_EMPTY_RULE",
                    "diff_count": 0,
                    "message": "两侧均无有效输入，跳过",
                }
            )
            continue

        for standard_topic, new_topic in topic_mapping.items():
            rule_topic_count += 1

            if available_topics is not None and standard_topic not in available_topics:
                logger.warning(
                    "标准topic不存在，跳过: rule_id=%s, standard_topic=%s, new_topic=%s",
                    rule_id,
                    standard_topic,
                    new_topic,
                )
                continue

            if available_topics is not None and new_topic not in available_topics:
                logger.warning(
                    "新topic不存在: rule_id=%s, standard_topic=%s, new_topic=%s",
                    rule_id,
                    standard_topic,
                    new_topic,
                )

            logger.info(
                "开始比对: rule_id=%s, standard_topic=%s, new_topic=%s, "
                "input_cid1=%s, input_uid1=%s, input_cid2=%s, input_uid2=%s",
                rule_id,
                standard_topic,
                new_topic,
                input_cid1 or "<empty>",
                input_uid1 or "<empty>",
                input_cid2 or "<empty>",
                input_uid2 or "<empty>",
            )

            try:
                # 从全局索引中按规则提取该topic的最新命中消息（仅使用 UID）。
                std_msg = get_latest_message_from_index(
                    std_indexes.get(standard_topic), input_uid1
                )
                new_msg = get_latest_message_from_index(
                    new_indexes.get(new_topic), input_uid2
                )

                # 命中后才反序列化 value bytes；解析失败视为该侧未找到。
                std_payload = _parse_payload_bytes(std_msg["payload"]) if std_msg else None
                new_payload = _parse_payload_bytes(new_msg["payload"]) if new_msg else None
                if std_payload is None:
                    std_msg = None
                if new_payload is None:
                    new_msg = None

                # 缺失场景也属于结果的一部分，按业务要求写入报告。
                if std_msg is None and new_msg is None:
                    logger.info(
                        "比对结果: rule_id=%s, standard_topic=%s, new_topic=%s, status=BOTH_NOT_FOUND",
                        rule_id,
                        standard_topic,
                        new_topic,
                    )
                    summary_rows.append(
                        {
                            "rule_id": rule_id,
                            "input_cid1": input_cid1,
                            "input_cid2": input_cid2,
                            "input_uid1": input_uid1,
                            "input_uid2": input_uid2,
                            "standard_topic": standard_topic,
                            "new_topic": new_topic,
                            "status": "BOTH_NOT_FOUND",
                            "diff_count": 0,
                            "message": "标准topic与新topic均未找到匹配消息",
                        }
                    )
                    continue

                if std_msg is not None and new_msg is None:
                    logger.info(
                        "比对结果: rule_id=%s, standard_topic=%s, new_topic=%s, status=MISSING_IN_NEW_TOPIC",
                        rule_id,
                        standard_topic,
                        new_topic,
                    )
                    summary_rows.append(
                        {
                            "rule_id": rule_id,
                            "input_cid1": input_cid1,
                            "input_cid2": input_cid2,
                            "input_uid1": input_uid1,
                            "input_uid2": input_uid2,
                            "standard_topic": standard_topic,
                            "new_topic": new_topic,
                            "status": "MISSING_IN_NEW_TOPIC",
                            "diff_count": 1,
                            "message": "标准topic找到消息，但新topic未找到匹配消息",
                        }
                    )
                    diff_rows.append(
                        {
                            "rule_id": rule_id,
                            "input_cid1": input_cid1,
                            "input_cid2": input_cid2,
                            "input_uid1": input_uid1,
                            "input_uid2": input_uid2,
                            "standard_topic": standard_topic,
                            "new_topic": new_topic,
                            "字段路径": "$",
                            "差异类型": "新topic缺失消息",
                            "JSON1值": json.dumps(std_payload, ensure_ascii=False)[:500],
                            "JSON2值": "",
                            "JSON1类型": "object",
                            "JSON2类型": "",
                            "额外信息": f"standard_timestamp={std_msg['timestamp']}, offset={std_msg['offset']}",
                        }
                    )
                    continue

                if std_msg is None and new_msg is not None:
                    logger.info(
                        "比对结果: rule_id=%s, standard_topic=%s, new_topic=%s, status=MISSING_IN_STANDARD_TOPIC",
                        rule_id,
                        standard_topic,
                        new_topic,
                    )
                    summary_rows.append(
                        {
                            "rule_id": rule_id,
                            "input_cid1": input_cid1,
                            "input_cid2": input_cid2,
                            "input_uid1": input_uid1,
                            "input_uid2": input_uid2,
                            "standard_topic": standard_topic,
                            "new_topic": new_topic,
                            "status": "MISSING_IN_STANDARD_TOPIC",
                            "diff_count": 1,
                            "message": "新topic找到消息，但标准topic未找到匹配消息",
                        }
                    )
                    diff_rows.append(
                        {
                            "rule_id": rule_id,
                            "input_cid1": input_cid1,
                            "input_cid2": input_cid2,
                            "input_uid1": input_uid1,
                            "input_uid2": input_uid2,
                            "standard_topic": standard_topic,
                            "new_topic": new_topic,
                            "字段路径": "$",
                            "差异类型": "标准topic缺失消息",
                            "JSON1值": "",
                            "JSON2值": json.dumps(new_payload, ensure_ascii=False)[:500],
                            "JSON1类型": "",
                            "JSON2类型": "object",
                            "额外信息": f"new_timestamp={new_msg['timestamp']}, offset={new_msg['offset']}",
                        }
                    )
                    continue

                # 两边都找到，进行JSON比对
                differences = compare_payloads(std_payload, new_payload, DEFAULT_EXCLUDED_PATHS)
                status = "MATCHED" if len(differences) == 0 else "DIFF_FOUND"
                logger.info(
                    "比对结果: rule_id=%s, standard_topic=%s, new_topic=%s, status=%s, diff_count=%s",
                    rule_id,
                    standard_topic,
                    new_topic,
                    status,
                    len(differences),
                )
                summary_rows.append(
                    {
                        "rule_id": rule_id,
                        "input_cid1": input_cid1,
                        "input_cid2": input_cid2,
                        "input_uid1": input_uid1,
                        "input_uid2": input_uid2,
                        "standard_topic": standard_topic,
                        "new_topic": new_topic,
                        "status": status,
                        "diff_count": len(differences),
                        "message": f"standard_ts={std_msg['timestamp']}, new_ts={new_msg['timestamp']}",
                    }
                )

                for diff in differences:
                    # 保留 diff 中的全部字段（含 uid1/Market/Brand/CID List 等上下文信息）
                    row = {
                        "rule_id": rule_id,
                        "input_cid1": input_cid1,
                        "input_cid2": input_cid2,
                        "input_uid1": input_uid1,
                        "input_uid2": input_uid2,
                        "standard_topic": standard_topic,
                        "new_topic": new_topic,
                    }
                    row.update(diff)
                    diff_rows.append(row)

            except Exception as exc:
                # 任一topic/规则执行异常都保留错误记录，避免吞错。
                logger.exception(
                    "比对异常: rule_id=%s, standard_topic=%s, new_topic=%s",
                    rule_id,
                    standard_topic,
                    new_topic,
                )
                summary_rows.append(
                    {
                        "rule_id": rule_id,
                        "input_cid1": input_cid1,
                        "input_cid2": input_cid2,
                        "input_uid1": input_uid1,
                        "input_uid2": input_uid2,
                        "standard_topic": standard_topic,
                        "new_topic": new_topic,
                        "status": "ERROR",
                        "diff_count": 0,
                        "message": f"{type(exc).__name__}: {exc}",
                    }
                )

        logger.info(
            "规则执行完成: rule_id=%s, processed_topic_pairs=%s, "
            "input_cid1=%s, input_uid1=%s, input_cid2=%s, input_uid2=%s",
            rule_id,
            rule_topic_count,
            input_cid1 or "<empty>",
            input_uid1 or "<empty>",
            input_cid2 or "<empty>",
            input_uid2 or "<empty>",
        )

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    report_file = REPORT_DIR / f"json_compare_batch_report_{timestamp}.xlsx"
    generate_batch_report(summary_rows, diff_rows, report_file)
    status_counts: Dict[str, int] = {}
    for row in summary_rows:
        status = str(row.get("status", "UNKNOWN"))
        status_counts[status] = status_counts.get(status, 0) + 1

    logger.info("批量比对完成: report=%s", report_file)
    logger.info("状态汇总: %s", status_counts)

In [0]:
# start comparing
main()
